# Vecka 6 – Kapitel 6: kod

Koduppgifterna 7–8 (6 = avskrift av kapitlets exempel). Uppgift 9 kräver ett Kaggle-dataset – se not sist.

## Uppgift 7 – K-means på housing.csv

**a)** Koden väljer tre kolumner (`median_income`, `latitude`, `longitude`), kör K-means med 6 kluster och färgar en karta (long/lat) efter kluster – dvs geografiska inkomstgrupper.

**d)** Här väljer vi antal kluster med *inertia* (elbow) och *silhouette*. K-means är känslig för skala, så vi standardiserar först.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv("../dataset/housing.csv")
X = df[["median_income", "latitude", "longitude"]]
Xs = StandardScaler().fit_transform(X)

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    print(f"k={k}: inertia {km.inertia_:8.0f}   silhouette {silhouette_score(Xs, km.labels_):.3f}")

In [ ]:
# Karta med vald k (silhouette är högst vid k=2, men k=6 ger mer användbara segment)
km = KMeans(n_clusters=6, random_state=42, n_init=10)
X = X.copy()
X["Cluster"] = km.fit_predict(Xs).astype(str)
sns.relplot(x="longitude", y="latitude", hue="Cluster", data=X, height=6)
plt.show()

## Uppgift 8 – klusteranalys på hr_employee_data.xlsx

Nu finns ingen `y`: alla numeriska variabler är `X`. Vi droppar id och `left` (facit i kap 4) och ser om klustren ändå fångar uppsägningsbenägenhet.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

hr = pd.read_excel("../dataset/hr_employee_data.xlsx")
num = hr.select_dtypes("number").drop(columns=["Emp_Id", "left"], errors="ignore")
Xs = StandardScaler().fit_transform(num)

hr["kluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Xs)
print(hr.groupby("kluster")[list(num.columns) + ["left"]].mean().round(2).T)

**Tolkning.** Kluster 2 sticker ut: högst arbetsbelastning (fler projekt, ~237 tim/mån), högst betyg – och 35 % har slutat mot 6 % i kluster 0. Klustringen hittar alltså "överbelastade högpresterare" som riskgrupp, helt utan att titta på `left`. Användbart för HR att rikta insatser mot ett helt segment i stället för individer.

## Uppgift 9 – 1980s classic hits (Spotify)

Kräver nedladdning från Kaggle (auth): https://www.kaggle.com/datasets/thebumpkin/1980s-classic-hits-with-spotify-data

Lägg csv:n i `dataset/` och kör samma recept: välj numeriska ljud-features (tempo, energy, danceability …), `StandardScaler` → `KMeans`, välj k med silhouette. *Inte körd här – datasetet finns inte lokalt.*